In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from time import time
import anndata as ad
import scanpy as sc
import copy
from umap import UMAP
import leidenalg as la
from tysserand import tysserand as ty
from mosna import mosna as mosna

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["PL_TORCH_DISTRIBUTED_BACKEND"] = "gloo"




Préparation des données

Network

In [2]:
#fonctions pour récupérer facilement les éléments utiles à l'établissement du network

def get_nodes_pyarrow(fichier, col_attributes):

    df = pd.read_parquet(fichier, engine="pyarrow")

    cols = ['x_centroid', 'y_centroid', col_attributes]

    return df.loc[:, cols]

def get_nodes(fichier, col_attributes):
    #fichier = path vers fichier où il y a les nodes
    #col_attributes = str = nom de la colonne contenant les attributs , exemple col_attribute = 'Cluster'
    nodes = pd.read_parquet(fichier)
    
    nodes = pd.read_parquet(fichier, columns=['x_centroid','y_centroid', col_attributes])
    
    return nodes

def get_coords(fichier, nodes):
    #fichier = path vers fichier où il y a les nodes
    coords = nodes.loc[:,['x_centroid','y_centroid']].values
    return coords
    
def get_pairs(fichier, nodes, coords):
    #fichier = path vers fichier où il y a les nodes
    pairs = ty.build_delaunay(coords)

    #récupération des indices comme id car delaunay reset les indices et on perd correspondance
    #node_ids = nodes.index.to_numpy()
    #pairs_ids = node_ids[pairs]

    return pairs

def get_network_label(nodes, col_attributes):
    labels = nodes[col_attributes].to_numpy()
    #label = nodes[col_attributes].unique()
    return labels

In [ ]:
from networkx import nodes


def attributes_to_colors_cmap(nodes, color_labels, col_attributes):
#fonction qui sert à retourner une list où chaque attribut est remplacé une couleur en fonction de son label
#nodes = dataframe avec tous les nodes
#color_label = array de tous les atributs qui ont besoin d'une couleur
#col_attributes = str = nom de la colonne contenant les attributs , exemple col_attribute = 'Cluster'
    classes = nodes[col_attributes].unique().tolist()
    colormap = mosna.make_cluster_cmap(color_labels)
    dico_col = dict(zip(classes, colormap))
    colors = nodes[col_attributes].map(dico_col).tolist()
    return colors

def attributes_to_colors_dico(nodes, col_attributes):
    classes = sorted(nodes[col_attributes].unique())
    n_classes = len(classes)

    palette = sns.color_palette("tab20", n_colors=n_classes)
    dico_col = dict(zip(classes, palette))
    colors = nodes[col_attributes].map(dico_col).tolist()

    return colors


def network_plot_unique (nodes,coords, pairs, labels, colors,fichier, dossier_enregistrement_networks,linewidth=1):
    #construit un plot du network calculé grâce à delauna et l'enregistre
    #colors = list où chaque attribut est remplacé une couleur en fonction de son label (peut être consrtuit à l'aide de ma fonction attributes_to_colors
    #linewidth = int taille du edge sur le plot, par défaut linewidth = 1
    #fichier = path vers le fichier nodes
    #dossier_enregistrement_networks  path vers le directory où enregistrer les png des networks

    nom = Path(fichier).stem
    
    fig, ax = plt.subplots(figsize=(20, 15))

    ty.plot_network(coords=coords,
                    pairs=pairs,
                    labels=labels,
                    color_mapper = colors,
                    #legend= True,
                    alpha_edges = 0.5,
                    col_edges='w',
                    legend_opt = {'loc': 'center left', 'bbox_to_anchor': (1.03, 0.5),'markerscale' : 2},
                    size_nodes=1.5,
                    ax=ax,
                    linewidth=linewidth
                    )
    plt.subplots_adjust(right=0.8)
    ax.set_facecolor('black')
    ax.set_title(f"Network {nom}")
    ax.set_frame_on(True)
    

    enregistrement = os.path.join(dossier_enregistrement_networks,f'Network {nom}.png')
    
    plt.savefig(
    enregistrement,
    dpi=300,
    bbox_inches="tight"
   
)
    plt.close(fig)


def network_plot_dossier(dossier_nodes,dossier_edges,dossier_enregistrement_networks, col_attributes,colors_dico = None):
#construit un network grâce à triangulation delaunay pour chaque fichier d'un dossier et les enregistre dans dossier_enregistrement
#dossier_nodes = path dossier vers nodes
#dossier_enregistrement = path dossier vers lequel on veut enregistrer les networks
#dossier_edges = path dossier vers lequel on veut enregistrer les edges
##col_attributes = str = nom de la colonne contenant les attributs , exemple col_attribute = 'Cluster'
#colors_dico = dico avec pour chaque attribut sa couleur par défaut None et le construit sur attributes du premier fichier
# par défaut colors = None et est produit par la fonction
    
    for fichier in os.listdir(dossier_nodes):
        if not fichier.startswith("nodes"):
            continue
        fichier = os.path.join(dossier_nodes, fichier)
        nodes = get_nodes(fichier=fichier,col_attributes=col_attributes)
        coords = get_coords(fichier=fichier, nodes=nodes)
        pairs = get_pairs(fichier=fichier, nodes=nodes, coords=coords)
        labels = get_network_label(nodes=nodes,col_attributes=col_attributes)

        # construire dictionnaire une seule fois
       
        classes = sorted(nodes[col_attributes].unique())
        palette = sns.color_palette("tab20", n_colors=len(classes))
        colors_dico = dict(zip(classes, palette))

        colors = colors_dico
        #colors = nodes[col_attributes].map(colors_dico).tolist()


        #colors = attributes_to_colors_dico(nodes, col_attributes)
        
        
        #color_labels = nodes[col_attributes]
        #colors = attributes_to_colors(nodes, col_attributes)
#je génère le network et je l'enregistre
        network_plot_unique(nodes=nodes,coords=coords, pairs=pairs, labels=labels, colors=colors, fichier=fichier, dossier_enregistrement_networks=dossier_enregistrement_networks)
#je récupère les edges
        node_ids = nodes.index.to_numpy()
        pairs_ids = node_ids[pairs]
        edges = ty.pairs_to_df(pairs_ids)
        #enregistrer edges
        nom_fichier = Path(fichier).name
        fichier_edges = nom_fichier.replace('nodes','edges')
        chemin = os.path.join(dossier_edges, fichier_edges)
        edges.to_parquet(chemin)

    

In [5]:
dossier_nodes = Path("./OUTPUT/niches/results_niche/results_1000/reducer-umap_dim-2_nneigh-20_metric-manhattan_min_dist-0.0/clusterer-leiden_n_neighbors-15/OUTPUT_merged")
dossier_edges = Path("./OUTPUT/niches/tysserand/1000/edges_tysserand/")
dossier_edges.mkdir(exist_ok=True)
dossier_enregistrement_networks = Path("./OUTPUT/niches/tysserand/1000/networks_tysserand/")
dossier_enregistrement_networks.mkdir(exist_ok=True)
col_attributes = 'Niche'
network_plot_dossier(dossier_nodes= dossier_nodes,
                    dossier_edges=dossier_edges,
                    dossier_enregistrement_networks=dossier_enregistrement_networks,
                    col_attributes=col_attributes
                    )

/home/mohamed.tahounza/miniconda3/envs/mosna/lib/python3.10/site-packages/tysserand/tysserand.py:1267: UserWarning: *c* argument looks like a single numeric RGB or RGBA sequence, which should be avoided as value-mapping will have precedence in case its length matches with *x* & *y*.  Please use the *color* keyword-argument or provide a 2D array with a single row if you intend to specify the same RGB or RGBA value for all points.
  ax.scatter(coords[select,0], coords[select,1], c=color, label=label,


ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

Assortativité

In [ ]:

save_dir = Path(dossier,"assortativity")
save_dir.mkdir(exist_ok=True)
net_dir = Path(dossier,'nodes_phenotyped_et_edges')

In [ ]:
#je récupère all_niches qui est une liste str de toutes les niches uniques
all_niches = set()

for f in net_dir.glob("nodes_*"):
    df = pd.read_parquet(f)
    all_niches.update(df["Niche"].astype(str).unique())

all_niches = sorted(all_niches)
print(len(all_niches))

all_niches

In [ ]:
#écrase chaque fichier en remplaçant par un onehot des niches en mettant dans chaque dataframe toutes les niches même si 
# non présentes de base dedans, afin d'avoir des dataframes "standardisés", sinon la fonction ne fonctionne pas
for f in net_dir.glob("nodes_*"):
    df = pd.read_parquet(f)

    df["Niche"] = pd.Categorical(df["Niche"], categories=all_niches)

    df = pd.get_dummies(df["Niche"],prefix='', prefix_sep='')

    df.to_parquet(f)

In [ ]:
attributes_col = [col for col in test.columns if col.startswith("Niche_")]
use_attributes = all_niches

In [ ]:
#c'est les paramètres pour calculer asortativity avec mosna.groups_assort_mixmat
save_dir = Path(dossier,"assortativity")
save_dir.mkdir(exist_ok=True)
net_dir = Path(dossier,'nodes_labeled_et_edges')
#net_dir doit contenir nodes et edges dans au même endroit
attributes_col = attributes_col
use_attributes = all_niches
make_onehot= False
id_level_1 = 'patient'
id_level_2 = 'sample'
memory_limit = '60GB'
n_shuffle = 150
extension = 'parquet'
parallel_groups= 'max'

stats_data = mosna.groups_assort_mixmat(net_dir= net_dir, attributes_col= attributes_col, use_attributes= use_attributes, make_onehot = make_onehot, id_level_1=id_level_1, id_level_2=id_level_2,extension=extension,parallel_groups=parallel_groups, n_shuffle=n_shuffle, memory_limit=memory_limit)
                                      


In [ ]:
stats_data